# 07 — Quantum Frequency Combs

**Engineering question**

How does a single integrated microresonator support many stable, addressable optical frequency modes?

Notebook 05 established frequency multiplexing as the replacement for replicated hardware.  
Notebook 07 explains the photonic object that makes that strategy useful:

```text
integrated resonator
→ discrete resonances
→ fixed mode spacing
→ frequency comb
→ many addressable optical channels
```

This notebook stays visual and architectural.  
The nonlinear physics that generates new comb lines is introduced here, then handled more directly in Notebook 11.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/07_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Resonator modes

A microresonator supports discrete resonances because only certain circulating fields satisfy the round-trip phase condition.

For this notebook, the important idea is:

```text
one physical cavity
→ many allowed optical modes
```

The figure below uses mode labels around the resonator to emphasize that the cavity is a physical device supporting a ladder of optical resonances.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 600)

fig, ax = plt.subplots(figsize=(6.2, 6.2))

# ring
ax.plot(np.cos(theta), np.sin(theta), linewidth=3)
ax.plot(0.72 * np.cos(theta), 0.72 * np.sin(theta), linewidth=1.5, alpha=0.35)

# bus waveguide
ax.plot([-1.55, 1.55], [-1.35, -1.35], linewidth=2.5)
ax.annotate("", xy=(1.5, -1.35), xytext=(1.05, -1.35), arrowprops=dict(arrowstyle="->", linewidth=2))

# many resonance labels around ring
mode_labels = ["ω₀", "ω₊₁", "ω₊₂", "ω₊₃", "ω₊₄", "ω₊₅", "ω₋₅", "ω₋₄", "ω₋₃", "ω₋₂", "ω₋₁"]
angles = np.linspace(90, -270, len(mode_labels), endpoint=False)

for ang, label in zip(angles, mode_labels):
    t = np.deg2rad(ang)
    ax.scatter([0.98 * np.cos(t)], [0.98 * np.sin(t)], s=35)
    ax.text(1.23 * np.cos(t), 1.23 * np.sin(t), label, ha="center", va="center", fontsize=10)

ax.text(0, 0, "integrated\nmicroresonator", ha="center", va="center", fontsize=12)
ax.text(0, -1.62, "bus waveguide", ha="center", fontsize=10)

ax.set_title("One Resonator Supports Many Discrete Optical Modes", fontsize=16)
ax.set_aspect("equal")
ax.set_xlim(-1.7, 1.7)
ax.set_ylim(-1.75, 1.55)
ax.axis("off")

savefig(fig, "07_ring_resonator_modes.png")
plt.show()


## 2. Frequency comb

A frequency comb is a set of regularly spaced optical resonances:

```text
ωₙ = ω₀ + nΔω
```

where \(Δω\) is the free spectral range, often abbreviated FSR.

Here, every line is a potential addressable frequency channel.


In [ ]:
n = np.arange(-10, 11)
envelope = np.exp(-(n / 7.5) ** 2)
heights = 0.35 + 0.65 * envelope

fig, ax = plt.subplots(figsize=(11, 4.0))

for ni, h in zip(n, heights):
    linewidth = 3.0 if ni == 0 else 2.0
    ax.plot([ni, ni], [0, h], linewidth=linewidth)
    ax.scatter([ni], [h], s=85 if ni == 0 else 60)

ax.axvline(0, linestyle="--", linewidth=1.4, alpha=0.65)
ax.text(0, 1.14, "pump / center mode\nω₀", ha="center", fontsize=10)

# spacing annotation
ax.annotate("", xy=(0, 0.22), xytext=(1, 0.22), arrowprops=dict(arrowstyle="<->", linewidth=1.6))
ax.text(0.5, 0.29, "Δω = FSR", ha="center", fontsize=10)

# symmetric sideband annotation
ax.annotate("symmetric sidebands", xy=(-4, heights[n.tolist().index(-4)]), xytext=(-7.5, 1.15),
            arrowprops=dict(arrowstyle="->", linewidth=1.2), fontsize=10)
ax.annotate("", xy=(4, heights[n.tolist().index(4)]), xytext=(6.9, 1.12),
            arrowprops=dict(arrowstyle="->", linewidth=1.2))
ax.text(0, -0.24, "each line can be addressed as a frequency channel", ha="center", fontsize=11)

ax.set_title("Frequency Comb: Equally Spaced Optical Modes", fontsize=16)
ax.set_xlabel("Mode index n")
ax.set_ylabel("Resonance")
ax.set_xticks([-10, -5, 0, 5, 10])
ax.set_xticklabels(["ω₀−10Δω", "ω₀−5Δω", "ω₀", "ω₀+5Δω", "ω₀+10Δω"])
ax.set_yticks([])
ax.set_xlim(-10.8, 10.8)
ax.set_ylim(-0.32, 1.38)
ax.grid(True, axis="x", alpha=0.12)

savefig(fig, "07_frequency_comb.png")
plt.show()


## 3. Comb generation pipeline

A pump laser couples into a resonator.

The resonator provides strong field confinement and discrete modes.

Nonlinear Kerr mixing then provides the mechanism that can create additional frequencies.

This notebook names that step.  
Notebook 11 models it more directly.


In [ ]:
labels = [
    "Pump\nlaser",
    "Ring\nresonator",
    "χ³ Kerr\nnonlinear mixing",
    "Frequency\ncomb",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(11, 3.0))

for xi, label in zip(x, labels):
    rect = plt.Rectangle((xi - 0.42, -0.24), 0.84, 0.48, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=11)

for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.58, 0), xytext=(i + 0.42, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

ax.set_title("Comb Generation Pipeline", fontsize=16)
ax.set_xlim(-0.7, len(labels) - 0.3)
ax.set_ylim(-0.65, 0.65)
ax.axis("off")

savefig(fig, "07_comb_generation_pipeline.png")
plt.show()


## 4. Fixed mode spacing

The engineering invariant is fixed mode spacing.

A simple idealized comb has:

```text
ωₙ = ω₀ + nΔω
```

So plotting frequency against mode index gives a straight line.

That fixed spacing is what makes the comb a usable frequency ruler and multiplexing resource.


In [ ]:
n = np.arange(-12, 13)
omega0 = 193.5
FSR = 0.1
omega = omega0 + n * FSR

fig, ax = plt.subplots(figsize=(7.2, 5.0))

ax.plot(n, omega, "o-", linewidth=2.4)
ax.set_title("Fixed Mode Spacing: ωₙ = ω₀ + nΔω", fontsize=16)
ax.set_xlabel("Mode index n")
ax.set_ylabel("Frequency (arb.)")
ax.grid(True, alpha=0.3)

# annotate equal increments
for ni in [-8, -2, 4]:
    ax.annotate("", xy=(ni + 1, omega0 + (ni + 1) * FSR), xytext=(ni, omega0 + ni * FSR),
                arrowprops=dict(arrowstyle="<->", linewidth=1.1))
ax.text(-8.6, omega0 + (-7.2) * FSR, "same Δω", fontsize=10)

savefig(fig, "07_mode_spacing.png")
plt.show()


## 5. Hardware replication versus frequency resources

The resource shift is the same one established in Notebook 05, but now the right-hand side is a comb.

The scaling variable shifts from:

```text
number of physical devices
```

to:

```text
number of addressable resonator modes
```


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))

# Replicated devices
for i in range(4):
    y = 3.5 - i * 0.85
    rect = plt.Rectangle((0.3, y - 0.25), 0.85, 0.5, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(0.725, y, f"Device {i+1}", ha="center", va="center", fontsize=10)

ax.text(0.72, 4.05, "Replication", ha="center", fontsize=14, weight="bold")
ax.text(0.72, 0.15, "one device → one channel", ha="center", fontsize=11)

# Arrow
ax.annotate("", xy=(2.8, 2.25), xytext=(1.55, 2.25), arrowprops=dict(arrowstyle="->", linewidth=2.5))
ax.text(2.15, 2.46, "scaling variable\nshifts", ha="center", fontsize=10)

# Resonator
circle = plt.Circle((3.75, 2.25), 0.55, fill=False, linewidth=2.5)
ax.add_patch(circle)
ax.text(3.75, 2.25, "integrated\nresonator", ha="center", va="center", fontsize=11)

# Comb lines
mode_x = np.linspace(5.5, 9.3, 10)
for i, mx in enumerate(mode_x):
    ax.plot([mx, mx], [1.15, 3.3], linewidth=2)
    ax.scatter([mx], [3.3], s=65)
    ax.text(mx, 0.9, f"ω{i+1}", ha="center", fontsize=8)

ax.text(7.4, 4.05, "Frequency-comb resource", ha="center", fontsize=14, weight="bold")
ax.text(7.4, 0.15, "one resonator → many addressable modes", ha="center", fontsize=11)

ax.set_title("Hardware Replication versus Frequency Resources", fontsize=16)
ax.set_xlim(0, 10.1)
ax.set_ylim(-0.05, 4.45)
ax.axis("off")

savefig(fig, "07_resource_shift.png")
plt.show()


## 6. Engineering summary

The contrast is not between a laser and a comb in the abstract.

It is between a **single-frequency source** and a **frequency-comb resource**.

That wording matters because the scaling claim is about the number of addressable frequencies.


In [ ]:
summary = pd.DataFrame(
    {
        "Property": ["Frequencies", "Resonator", "Channels", "Spacing", "Scaling variable"],
        "Single-frequency source": ["one", "optional", "one", "single frequency", "devices"],
        "Frequency-comb resource": ["many", "essential", "many", "fixed FSR", "modes"],
    }
)

fig, ax = plt.subplots(figsize=(10.5, 3.0))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1, 1.65)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Single Frequency versus Frequency Comb", fontsize=16, pad=20)

savefig(fig, "07_summary.png")

summary.to_csv(CSV / "07_summary.csv", index=False)
summary.to_json(JS / "07_summary.json", orient="records", indent=2)

plt.show()
summary


## 7. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "07_outputs.zip"

files_to_zip = [
    FIG / "07_ring_resonator_modes.png",
    FIG / "07_frequency_comb.png",
    FIG / "07_comb_generation_pipeline.png",
    FIG / "07_mode_spacing.png",
    FIG / "07_resource_shift.png",
    FIG / "07_summary.png",
    CSV / "07_summary.csv",
    JS / "07_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- A microresonator supports discrete optical resonances.
- A frequency comb arranges those resonances into a fixed-spacing mode ladder.
- The comb changes the resource from replicated devices to addressable frequencies.
- The Kerr step has now been named but not yet modeled.

## Next notebook

**11 — Kerr Nonlinearity**

```text
pump laser
→ χ³ Kerr nonlinear mixing
→ new frequency components
→ comb generation mechanism
```

Notebook 11 should answer:

```text
How does nonlinear interaction turn a pump field into new frequency modes?
```
